# Civic Deserts: Mapping Service Access Across South Africa's Municipal Districts

**Where do people live far from the services they depend on?**

This notebook ranks South Africa's 52 district municipalities and metros by how many people
each clinic, school, bank, and police station has to serve — and flags the districts that are
consistently underserved across multiple measures at once, not just one.

It combines three kinds of open data:

- **Population**: [HDX HAPI baseline population](https://data.humdata.org/dataset/hdx-hapi-population)
  (age/gender structure, 2020) and the [Stats SA Census 2022 Municipal fact sheet](https://www.statssa.gov.za)
  (current district totals)
- **Settlements**: [HOT OSM South Africa Populated Places](https://data.humdata.org/dataset/hotosm_zaf_populated_places)
- **Amenities**: HOT OSM exports for health facilities, education facilities, financial services,
  and the general points-of-interest layer (for police stations)
- **Boundaries**: the OCHA Common Operational Dataset for South African administrative boundaries

This notebook walks through the data, the two real data-quality bugs we found and fixed along the
way, the population methodology, and the final ranking. The companion repository
([Civic_Deserts](https://github.com/Lindiwe-22/Civic_Deserts)) has the full pipeline as reusable scripts;
this notebook tells the story and shows the work.

## 1. Data sources and collection

Four amenity classes were chosen to represent core civic services people rely on locally:

| Class | Definition | Source |
|---|---|---|
| Clinics | `amenity=clinic` or `amenity=hospital` | HOT health facilities export |
| Schools | `amenity=school` | HOT education facilities export |
| Banks | `amenity=bank` or `amenity=atm` | HOT financial services export |
| Police | `amenity=police` | HOT general points-of-interest export (no dedicated police export exists) |

Population combines two sources deliberately rather than picking one — see Section 4 for why.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"

pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

In [ ]:
ranking = pd.read_csv(PROCESSED / "district_ranking.csv")
population = pd.read_csv(PROCESSED / "district_population_2022.csv")
amenities = pd.read_csv(PROCESSED / "district_amenity_counts.csv")
by_age = pd.read_csv(PROCESSED / "district_population_2022_by_age.csv")

print(f"{len(ranking)} districts, {ranking['population_2022'].sum():,.0f} total population")
ranking.head()

## 2. Exploratory data analysis

### 2.1 Population distribution

South Africa's district populations are highly skewed — a handful of metros hold millions of
people while rural districts hold a few hundred thousand. Any per-capita ratio needs to be read
with this in mind: a metro's raw amenity count will always look large, but that's expected given
its population, not evidence of good service.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(population["population_2022"], bins=20, color="#4C72B0", edgecolor="white")
axes[0].set_title("District population distribution")
axes[0].set_xlabel("Population (2022)")
axes[0].set_ylabel("Number of districts")

top10 = population.nlargest(10, "population_2022").sort_values("population_2022")
axes[1].barh(top10["display_name"], top10["population_2022"], color="#4C72B0")
axes[1].set_title("10 most populous districts")
axes[1].set_xlabel("Population (2022)")

plt.tight_layout()
plt.savefig("../outputs/figures/population_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.2 Amenity counts by class

Before computing any ratio, it's worth looking at the raw counts — how many of each amenity type
OpenStreetMap actually has mapped, nationally.

In [ ]:
amenity_totals = amenities[["n_clinic", "n_school", "n_bank", "n_police"]].sum()
print("National totals (after the fixes described in 2.3):")
print(amenity_totals.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
amenity_totals.plot(kind="bar", ax=ax, color=["#C44E52", "#55A868", "#4C72B0", "#8172B2"])
ax.set_title("Total amenities mapped nationally, by class")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../outputs/figures/amenity_totals.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.3 Data quality: two undercounting bugs, found and fixed

Building this pipeline surfaced two real data-quality issues worth documenting, since they changed
the results substantially and are a useful lesson in not trusting a single OSM tagging convention.

**Bug 1 — clinics mapped as building polygons, not points.** The dedicated HOT health facilities
export is points-only. Two districts (Ugu, Umgungundlovu) initially showed **zero** clinics, which
was implausible for districts of 773,000 and 1.2 million people. The cause: 14 and 23 clinics/hospitals
respectively were mapped as building *polygons* in OpenStreetMap's general points-of-interest layer,
invisible to a points-only query.

**Bug 2 — the same pattern, much larger, for schools.** The dedicated education export counted
only ~1,200 schools nationally — under 5% of South Africa's actual ~23,000–25,000 schools. Including
polygon-mapped schools from the general POI layer raised this to ~9,400, a far more plausible (if
still incomplete) figure. eThekwini alone went from 14 schools to 450.

**The fix**, applied consistently to clinics, schools, banks and police: combine points and polygon
centroids from every available source, then deduplicate anything within 50 meters of another match
(so a facility mapped twice, once as a point and once as a polygon, is only counted once).

This means the amenity counts in this analysis are still a **floor, not a ceiling** — OpenStreetMap
coverage, even after this fix, is incomplete. The relative ranking between districts is more trustworthy
than any single district's absolute count.

In [ ]:
print("Schools, before vs after including polygon-mapped facilities:")
print(f"  Points-only (dedicated export):  ~1,197")
print(f"  Points + polygons, deduplicated: {amenities['n_school'].sum():,}")
print()
print("Clinics, the district that first exposed the bug:")
print(f"  Ugu:            0 -> 13 clinics")
print(f"  Umgungundlovu:   0 -> 22 clinics")

## 3. Population methodology

District population combines two sources on purpose:

- **Totals** come from the **Stats SA Census 2022 Municipal fact sheet** — the more current,
  directly measured figure.
- **Age structure** comes from **HAPI's 2020 baseline**, which provides a 5-year age-band breakdown.
  Each district's 2020 age-band *shares* are applied to that district's 2022 Stats SA *total*, giving
  a current population level with 2020-era age structure.

This was a deliberate choice, not a default. Comparing the two sources directly showed HAPI's 2020
district-level *totals* diverging from Census 2022 in ways that don't track real demographic change —
e.g. City of Johannesburg showing a 13% population *decline* over two years, while several small rural
districts showed 30%+ growth. That pattern is far more consistent with HAPI's district split using
older population shares than with real two-year demographic shifts. Its age *proportions* are more
defensible to carry forward than its absolute district counts.

Four districts were renamed between when HAPI's admin boundaries were coded and the 2022 fact sheet's
publication; both names are kept for identifiability (e.g. "Sarah Baartman (Cacadu)").

In [ ]:
rename_check = population[population["display_name"].str.contains(r"\(", regex=True)]
print("Districts with a former name shown in parentheses:")
rename_check[["admin2_code", "display_name"]]

## 4. Building the desert score

For each district and each of the four amenity classes, we compute **people per amenity**:
population divided by facility count. A higher number means each facility is serving more people —
worse access.

Two more targeted ratios narrow the denominator to the population that actually uses that service:

- **School-age population (5–19) per school** — rather than total population
- **Older-adult population (65+) per clinic** — a population segment with disproportionate healthcare need

Finally, a composite **desert score** ranks districts by how *consistently* underserved they are: each
district's percentile rank on each of the four overall ratios, averaged. This means a district doesn't
need to be the single worst on any one measure to surface — being persistently in the worse half across
all four is enough to rank highly.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
ratio_cols = ["people_per_clinic", "people_per_school", "people_per_bank", "people_per_police"]
titles = ["People per clinic", "People per school", "People per bank", "People per police station"]

for ax, col, title in zip(axes.flat, ratio_cols, titles):
    top5 = ranking.nlargest(5, col).sort_values(col)
    ax.barh(top5["display_name"], top5[col], color="#C44E52")
    ax.set_title(title)
    ax.set_xlabel("People served per facility")

plt.tight_layout()
plt.savefig("../outputs/figures/worst_ratios_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Findings

### 5.1 The most underserved districts

These districts rank worst on the composite desert score — consistently high people-per-amenity
ratios across multiple classes, not just one outlier metric.

In [ ]:
display_cols = ["display_name", "population_2022", "n_clinic", "n_school", "n_bank",
                 "n_police", "people_per_clinic", "people_per_school",
                 "people_per_bank", "people_per_police", "desert_score"]
ranking.sort_values("desert_score", ascending=False)[display_cols].head(10)

Several of the worst-ranked districts are large, rural, and in Limpopo or the Eastern Cape
(Vhembe, Alfred Nzo, Sekhukhune), which is consistent with South Africa's well-documented rural-urban
service divide. But the composite score also surfaces metros — Buffalo City, eThekwini, Ekurhuleni —
driven by weak clinic and police ratios even where school and bank access look adequate, a reminder
that "underserved" isn't only a rural story.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
worst10 = ranking.nlargest(10, "desert_score").sort_values("desert_score")
ax.barh(worst10["display_name"], worst10["desert_score"], color="#C44E52")
ax.set_xlabel("Desert score (higher = more consistently underserved)")
ax.set_title("10 most underserved districts")
plt.tight_layout()
plt.savefig("../outputs/figures/desert_score_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2 Age-specific views

Looking at school-age and older-adult populations specifically changes the picture slightly —
worth checking against the overall ranking.

In [ ]:
print("School-age (5-19) per school, most stretched:")
display(ranking.nlargest(5, "school_age_per_school")[
    ["display_name", "population_5_19", "n_school", "school_age_per_school"]])

print("\nOlder-adult (65+) per clinic, most stretched:")
display(ranking.nlargest(5, "older_adult_per_clinic")[
    ["display_name", "population_65plus", "n_clinic", "older_adult_per_clinic"]])

### 5.3 The best-served districts, for contrast

In [ ]:
ranking.sort_values("desert_score")[display_cols].head(10)

## 6. Limitations and next steps

- **OSM completeness is a floor, not a ceiling.** Even after fixing the polygon-undercounting bug,
  OpenStreetMap coverage of schools in particular is likely still well under 50% complete nationally.
  Every district's *true* amenity access is probably better than shown here — but there's no strong
  reason to believe the undercount is systematically biased toward or against any particular district,
  so the *relative* ranking should be more trustworthy than any single absolute count.
- **Population vintage mismatch.** Age-band structure reflects 2020 (HAPI); totals reflect 2022
  (Stats SA). Two years of demographic shift between them is assumed small relative to the benefit of
  having both current totals and detailed age structure.
- **District-level averaging hides within-district variation.** A district with an adequate average
  ratio can still contain underserved pockets — dense townships or remote settlements far from the
  district's amenities, which happen to be concentrated near the district's urban core. This is exactly
  what the next phase of this project — a settlement-level distance analysis — is designed to surface.
- **Census 2022 undercounting.** Stats SA's 2022 Census faced public criticism over undercounting in
  some metros, particularly Gauteng. Neither population source here should be treated as unquestionably
  correct.

**Next**: mapping `desert_score` as a choropleth across the 52 districts, and a settlement-level
distance-to-nearest-amenity layer using the populated places data, independent of district-level
population weighting.

---
*Full pipeline, scripts and data notes: [github.com/Lindiwe-22/Civic_Deserts](https://github.com/Lindiwe-22/Civic_Deserts)*